# Structured output

- Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

 **i.e.** 

   - Structured output means asking an AI model to give its answer in a **specific format or structure** instead of plain text.  
   -  This makes the result **easy to read, organize, and process automatically** in programs.
   - For example, instead of a paragraph, the model might return data in a format like JSON with clearly labeled fields.  
   - Frameworks like LangChain help enforce these predefined structures so the output follows the expected schema.
   ---
![ChatGPT Image Mar 10, 2026, 06_42_13 PM (1).png](<attachment:ChatGPT Image Mar 10, 2026, 06_42_13 PM (1).png>)

# **Pydantic**

- Pydantic is a Python library used for data validation and settings management using Python type hints. It allows you to define the expected structure of your data using standard Python classes and type annotations, and Pydantic will automatically validate incoming data against this structure. It provides features like automatic data parsing, validation, and serialization, making it widely used in web development, APIs, and data processing pipelines.
- Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.


In [2]:
# STEP 1 : Model Initialization
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq
model = ChatGroq(model="qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000200856844D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020085800AD0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
# STEP 2 : Schema Design / Create Schema

from pydantic import BaseModel,Field
from typing import List


class Movie(BaseModel):
    title : str = Field(description="The title of the movie"),
    year : int = Field(description="The year of release"),
    director : str = Field(description="The director of the movie"),
    rating : float = Field(description="The rating of the movie out of 10"),
    genre : List[str] = Field(description="The genre of the movie"),
    cast : List[str] = Field(description="The cast of the movie")

In [4]:
# STEP 3 : Apply the Schema in Model
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output

c:\Users\HP\Desktop\Study_for_Placement\Generative_AI_Krish_Naik\Gen_AI_Crash_Course_3HR\.venv\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\HP\Desktop\Study_for_Placement\Generative_AI_Krish_Naik\Gen_AI_Crash_Course_3HR\.venv\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The year of release'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\HP\Desktop\Study_for_Placement\Generative_AI_Krish_Naik\Gen_AI_Crash_Course_3HR\.venv\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default val

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000200856844D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020085800AD0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'type': 'string'}, 'year': {'type': 'integer'}, 'director': {'type': 'string'}, 'rating': {'type': 'number'}, 'genre': {'items': {'type': 'string'}, 'type': 'array'}, 'cast': {'description': 'The cast of the movie', 'items': {'type': 'string'}, 'type': 'array'}}, 'required': ['cast'], 'type': 'object'}}}], 'ls_structured_o

---
`# **Understanding PydanticToolsParser(first_tool_only=True, tools=[<class '__main__.Movie'>])** `


- **Purpose:** It is a LangChain **output parser** that converts the LLM’s response into a **Pydantic model** (here, `Movie`) and validates it.  

- **`tools=[<class '__main__.Movie'>]`:** Specifies that the available schema/tool for structured output is your `Movie` Pydantic model. LangChain treats the Pydantic model like a **tool** so the LLM can generate output matching it.  

- **`first_tool_only=True`:** Ensures that **only one structured object** is returned, not a list of objects.  

- **`with_structured_output(Movie)`:** 
  1. Converts the `Movie` Pydantic model to a **JSON schema**.  
  2. Sends the schema to the LLM to guide output.  
  3. Uses `PydanticToolsParser` to **validate and convert** the response into a Pydantic object.  

- **Conceptual Flow:**

User Prompt  
↓  
LLM generates structured response  
↓  
PydanticToolsParser  
↓  
Validated Pydantic Object (Movie)

In [5]:
# invoke without Schema
model.invoke("Rrovide the detalis about the movie Titanic")

AIMessage(content='<think>\nOkay, so the user is asking for details about the movie Titanic. Let me start by recalling what I know about Titanic. It\'s a 1997 film directed by James Cameron, right? Starring Leonardo DiCaprio and Kate Winslet as Jack and Rose. The movie is a romance set against the real-life sinking of the RMS Titanic. I should mention the historical context since the Titanic was an actual ship that sank in 1912. \n\nThe plot is about two people from different social classes falling in love, but their story gets cut short by the ship\'s collision with an iceberg. The movie combines the fictional story with the real disaster. I need to highlight the director, main actors, release year, and maybe the critical reception. Also, it\'s known for its visual effects and the soundtrack, especially the song "My Heart Will Go On" by Celine Dion.\n\nI should check if there are any notable awards. I remember it won 11 Oscars, which is a record. The budget was really high for its tim

In [6]:
# invoke with Schema
model_with_structured_output.invoke("Rrovide the detalis about the movie Titanic")

Movie(title='Titanic', year=1997, director='James Cameron', rating=7.8, genre=['Drama', 'Romance'], cast=['Leonardo DiCaprio', 'Kate Winslet', 'Billy Zane', 'Kathy Bates', 'Johnny Depp'])

### **Message output alongside Parsed structure**


- Means the **LLM returns two outputs at the same time**.
- One output is the **normal text message** generated by the model.
- The second output is the **parsed structured data** following a defined schema.
- The **message output** is useful for **human-readable responses**.
- The **parsed structure** is useful for **programmatic use (code, APIs, databases)**.
- Structured output usually follows a **schema** (e.g., title, content, conclusion).
- Common tools for defining schemas:
  - **Pydantic** → Field validation and type enforcement
  - **TypedDict** → Lightweight type structure
  - **Dataclasses** → Simple structured data models
- Helps avoid **manual parsing of LLM text responses**.
- Makes LLM outputs **reliable, structured, and easy to integrate into applications**.
- Widely used in **LangChain structured output pipelines**.

In [21]:
from pydantic import BaseModel,Field
from typing import List


class Movie(BaseModel):
    """A movie withdetails."""
    title : str = Field(description="The title of the movie"),
    year : int = Field(description="The year of release"),
    director : str = Field(description="The director of the movie"),
    rating : float = Field(description="The rating of the movie out of 10"),
    genre : List[str] = Field(description="The genre of the movie"),
    cast : List[str] = Field(description="The cast of the movie")

model_with_structured_output = model.with_structured_output(Movie,include_raw=True)
response = model_with_structured_output.invoke("Provide the detalis about the movie Titanic")
response

c:\Users\HP\Desktop\Study_for_Placement\Generative_AI_Krish_Naik\Gen_AI_Crash_Course_3HR\.venv\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\HP\Desktop\Study_for_Placement\Generative_AI_Krish_Naik\Gen_AI_Crash_Course_3HR\.venv\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The year of release'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\HP\Desktop\Study_for_Placement\Generative_AI_Krish_Naik\Gen_AI_Crash_Course_3HR\.venv\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default val

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Titanic. Let me see what I need to do here. The tools provided include a Movie function with parameters like cast, director, genre, rating, title, and year. The required parameter is cast, which is an array of strings.\n\nFirst, I need to check if I have all the necessary information. Since I don\'t have access to a database or external sources, I have to rely on my existing knowledge. I remember that Titanic is a 1997 film directed by James Cameron. The main cast includes Leonardo DiCaprio and Kate Winslet. The genres are romance and drama. The rating is pretty high, maybe around 7.8 on IMDb. \n\nWait, the required field is cast, so I need to make sure that\'s included. The director is James Cameron. The year is 1997. Let me confirm the genres again—yes, it\'s romance and drama. The rating might be a number, but I should check if it\'s a number type. The title is

---
`# Example Output Explanation (include_raw=True) :`

**When running:**

model_with_structured_output = model.with_structured_output(Movie, include_raw=True)

response = model_with_structured_output.invoke(
"Provide the details about the movie Titanic"
)

The model returns **three main parts**.



`1️⃣ raw`

`raw` contains the **original AIMessage generated by the LLM** before parsing.

**Important things inside raw:**

- **reasoning_content** → Model's internal reasoning process
- **tool_calls** → The structured function call generated by the model
- **response_metadata** → Token usage, model name, finish reason, etc.
- **tool_calls.args** → The structured JSON output produced by the model

`Example inside raw:`

{
"name": "Movie",
"args": {
"title": "Titanic",
"year": 1997,
"director": "James Cameron",
"rating": 7.8,
"genre": ["Romantic", "Drama"],
"cast": [
"Leonardo DiCaprio",
"Kate Winslet",
"Billy Zane",
"Kathy Bates"
]
}
}


`2️⃣ parsed`

`parsed` is the **validated Pydantic object created from the schema**.

Movie(
title='Titanic',
year=1997,
director='James Cameron',
rating=7.8,
genre=['Romantic','Drama'],
cast=['Leonardo DiCaprio','Kate Winslet','Billy Zane','Kathy Bates']
)

This is the **clean structured output ready for use in Python code**.



`3️⃣ parsing_error`

`parsing_error` shows whether the schema parsing failed.

parsing_error = None

Meaning:
- The model output **matched the Movie schema successfully**.



`# Complete Flow`

User Prompt  
↓  
LLM generates structured function output  
↓  
Raw AIMessage returned  
↓  
LangChain parses JSON into Pydantic schema  
↓  
Final structured Python object



`# Key Idea`

include_raw=True allows you to access:

- **raw → original LLM response**
- **parsed → structured schema object**
- **parsing_error → validation status**

This helps with **debugging, validation, and structured AI pipelines**.


---

### **Nested Structure :**
- **Nested structure** means a **structure inside another structure**.
- It is commonly used in **JSON, dictionaries, and Pydantic schemas**.
- One object contains **another object or list of objects** as its field.
- Helps represent **complex relationships in data**.
- Frequently used in **LLM structured outputs and APIs**.


In [22]:
from pydantic import BaseModel,Field
from typing import List

class Actor(BaseModel):
    """An actor with name and role."""
    name : str = Field(description="The name of the actor")
    role : str = Field(description="The role of the actor")

class MovieDetails(BaseModel):
    """A movie withdetails."""
    title : str = Field(description="The title of the movie")
    year : int = Field(description="The year of release")
    director : str = Field(description="The director of the movie")
    rating : float = Field(description="The rating of the movie out of 10")
    genre : List[str] = Field(description="The genre of the movie")
    cast : List[Actor] = Field(description="The cast of the movie")
    budget : float | None = Field(None,description="Budget in million USD")

model_with_structured_output = model.with_structured_output(MovieDetails)
response = model_with_structured_output.invoke("Provide the detalis about the movie Titanic")
response

MovieDetails(title='Titanic', year=1997, director='James Cameron', rating=7.8, genre=['Romance', 'Drama'], cast=[Actor(name='Leonardo DiCaprio', role='Jack'), Actor(name='Kate Winslet', role='Rose'), Actor(name='Billy Zane', role='Cal Hockley')], budget=200.0)

# **TypedDict**
- TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.


In [12]:
from typing_extensions import TypedDict,Annotated

In [17]:
# Create Schema
class Movie(TypedDict):
    """A movie withdetails."""
    title : Annotated[str,...,"The title of the movie"]
    year : Annotated[int,...,"The year of release"]
    director : Annotated[str,...,"The director of the movie"]
    rating : Annotated[float,...,"The rating of the movie out of 10"]
    genre : Annotated[List[str],...,"The genre of the movie"]
    cast : Annotated[List[str],...,"The cast of the movie"]


In [18]:
Movie

__main__.Movie

In [19]:
type(Movie)

typing_extensions._TypedDictMeta

In [ ]:
model_withtypedDict = model.with_structured_output(Movie)
response = model_withtypedDict.invoke("Provide the detalis about the movie Titanic")
response

# output in dictionary format

Movie(title='Titanic', year=1997, director='James Cameron', rating=7.8, genre=['Romance', 'Drama'], cast=['Leonardo DiCaprio', 'Kate Winslet', 'Billy Zane', 'Kathy Bates', 'Frances Fisher'])

In [25]:
# Nested structure example
class Actor(TypedDict):
    name : str
    role : str

class MovieDetails(TypedDict):
    title : str
    year : int
    director : str
    rating : float
    genre : List[str]
    cast : List[Actor]
    budget : float | None = Field(None,description="Budget in million USD")

model_with_structured_output = model.with_structured_output(MovieDetails)
response = model_with_structured_output.invoke("Provide the detalis about the movie Titanic")
response

{'budget': 200000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Jack Dawson'},
  {'name': 'Kate Winslet', 'role': 'Rose DeWitt Bukater'},
  {'name': 'Billy Zane', 'role': 'Caledon Hockley'},
  {'name': 'Kathy Bates', 'role': 'Molly Brown'}],
 'director': 'James Cameron',
 'genre': ['Romance', 'Drama'],
 'rating': 7.8,
 'title': 'Titanic',
 'year': 1997}

In [26]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

# **DataClasses**
- A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [28]:
# 1. Pydantic
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='2682cd99-dc50-4a53-929c-3e9856d454c6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text: "John Doe, john@example.com, (555) 123-4567". The tools provided include a ContactInfo function with parameters for name, email, and phone. All three are required.\n\nFirst, I need to parse the input. The name is "John Doe", which is straightforward. The email is "john@example.com". The phone number is "(555) 123-4567". I should check if the phone number is in the correct format. The function\'s parameters don\'t specify a particular format, so as long as it\'s a string, it should be acceptable. \n\nI need to make sure all required fields are present. The input has all three: name, email, phone. No missing data. So I can proceed to create the 

---
---
`# ContactInfo Extraction Example (Structured Agent Output) :`

**When running:**

agent = create_agent(
    model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
    }]
})

The agent returns **structured contact information based on the schema**.



`1️⃣ ContactInfo Schema`

The **ContactInfo Pydantic model** defines the structured output format.

class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

Each field includes a **description to guide the LLM**.



`2️⃣ Agent Processing`

The agent performs the following steps:

- Reads the **user message**
- Understands the **schema requirements**
- Extracts relevant information from the text
- Fills the schema fields accordingly



`3️⃣ Structured Output`

The final result is returned as a **validated Pydantic object**.

ContactInfo(
name="John Doe",
email="john@example.com",
phone="(555) 123-4567"
)

This ensures the output follows the **exact schema structure**.



`# Complete Flow`

User Input Text  
↓  
LLM reads schema (ContactInfo)  
↓  
Agent extracts structured data  
↓  
LangChain validates using Pydantic  
↓  
Final structured Python object returned



`# Key Idea`

Using **response_format=ContactInfo** allows:

- Automatic **information extraction**
- Reliable **structured output**
- Direct use in **Python programs, APIs, or databases**

This approach converts **unstructured text → structured data automatically**.

---

In [29]:
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [30]:
# 2. Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [32]:
# 3. Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')